# CNN Case Study: Multi-Class Image Classification

## Project Overview

In this case study, you will apply everything you've learned about CNNs to build a complete image classification pipeline. We'll use the **CIFAR-10 dataset** - a benchmark dataset containing 60,000 32x32 color images in 10 classes.

### Learning Objectives
- Build an end-to-end CNN image classification pipeline
- Apply data augmentation techniques
- Compare different CNN architectures
- Use transfer learning for improved performance
- Evaluate and interpret model results
- Handle real-world challenges (class imbalance, overfitting)

### Dataset: CIFAR-10
- **Source**: Canadian Institute For Advanced Research
- **Classes**: Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck
- **Size**: 60,000 images (50,000 training, 10,000 test)
- **Image Size**: 32x32 RGB

### Prerequisites
- Completed notebooks 01-05 in this CNN module
- Understanding of convolutions, pooling, and regularization

---

# CNN case study: CIFAR-10 image classification — Assessment (D023)

**Time:** 60 minutes · **Marks:** 100 · **Format:** coding notebook plus written justification, marked by a human.

**Data:** `tensorflow.keras.datasets.cifar10 (cached locally after the first download)`

**Rules:** only what has been taught up to this day; comment the *why*, not the *what*; a printed number without its justification earns half the marks.

Tasks 4, 6, 7, 11 and the four exercises are bonus. The instructor copy trains on a 10,000-image subset for 3 epochs so a CPU can produce it; the numbers are a reference run, not targets.

| Task | Bloom level | Marks | What is marked |
|---|---|---|---|
| Tasks 1 to 3: explore and preprocess | understand | 10 | Sample images shown with labels; class counts (balanced); pixels scaled to 0-1 and labels one-hot or sparse, with the shapes printed. |
| Task 5: baseline CNN | apply | 20 | Conv2D and MaxPooling2D blocks, Flatten, a Dense head with softmax; model.summary() read for the parameter count. |
| Task 8: callbacks and training | apply | 15 | EarlyStopping and a checkpoint; validation split; the training curves plotted. |
| Tasks 9 to 10: evaluation, confusion matrix, report | analyse | 20 | Test accuracy against the 10 percent chance rate; the confusion matrix read for the pair of classes most confused; precision and recall per class. |
| Task 12: error analysis | analyse | 15 | Misclassified images shown with true and predicted labels; a pattern named (cat vs dog, truck vs automobile). |
| Task 13: conclusions | evaluate | 20 | What the baseline learned and where it fails; what regularisation or transfer would change and at what cost; what more data would and would not fix. |

Anything headed **Bonus (not marked)** is optional stretch material.

## Part 1: Data Loading and Exploration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")

# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Class names
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"\nNumber of classes: {len(class_names)}")
print(f"Image shape: {X_train[0].shape}")
print(f"Pixel value range: [{X_train.min()}, {X_train.max()}]")

### Task 1: Visualize Sample Images

Create a visualization showing one sample image from each of the 10 classes.

In [ ]:
# YOUR CODE HERE
# Hint: Use matplotlib subplots to create a 2x5 grid
# For each class i, find an image where y_train == i



### Task 2: Check Class Distribution

Create a bar chart showing the number of samples per class in the training set.

In [ ]:
# YOUR CODE HERE
# Hint: Use np.unique with return_counts=True



## Part 2: Data Preprocessing

### Task 3: Preprocess the Data

Complete the following preprocessing steps:
1. Normalize pixel values to [0, 1]
2. One-hot encode the labels
3. Create a validation split (10% of training data)

In [ ]:
# YOUR CODE HERE
# Step 1: Normalize pixel values
X_train_norm = None  # Replace with your code
X_test_norm = None   # Replace with your code

# Step 2: One-hot encode labels
y_train_cat = None   # Replace with your code
y_test_cat = None    # Replace with your code

# Step 3: Create validation split
val_split = 0.1
val_size = int(len(X_train_norm) * val_split)

X_val = None          # Replace with your code
y_val = None          # Replace with your code
X_train_final = None  # Replace with your code
y_train_final = None  # Replace with your code

print(f"Training set: {X_train_final.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test_norm.shape[0]} samples")

## Bonus (not marked): Part 3: Data Augmentation

Data augmentation helps prevent overfitting by creating variations of our training images.

### Bonus (not marked): Task 4: Create Data Augmentation Pipeline

Create a data augmentation pipeline that includes:
- Horizontal flip
- Rotation (up to 10%)
- Zoom (up to 10%)
- Translation (up to 10%)

In [ ]:
# YOUR CODE HERE
# Hint: Use keras.Sequential with layers.RandomFlip, RandomRotation, etc.
data_augmentation = None  # Replace with your code


In [ ]:
# Visualize augmented images (run after creating data_augmentation)
plt.figure(figsize=(12, 4))
sample_img = X_train_norm[0:1]  # Single image with batch dimension

plt.subplot(1, 5, 1)
plt.imshow(sample_img[0])
plt.title('Original')
plt.axis('off')

for i in range(4):
    plt.subplot(1, 5, i+2)
    augmented = data_augmentation(sample_img)
    plt.imshow(augmented[0])
    plt.title(f'Augmented {i+1}')
    plt.axis('off')

plt.suptitle('Data Augmentation Examples')
plt.tight_layout()
plt.show()

## Part 4: Building CNN Models

You'll build and compare three approaches:
1. **Simple CNN** - Basic architecture (baseline)
2. **Deep CNN** - More layers with regularization
3. **Transfer Learning** - Using pre-trained MobileNetV2

### Task 5: Build Simple CNN (Baseline)

Create a simple CNN with:
- 3 convolutional blocks (Conv2D + MaxPooling)
- Filter sizes: 32 → 64 → 64
- 3x3 kernels with 'same' padding
- 2x2 max pooling
- Dense classifier with 64 units

In [ ]:
def build_simple_cnn():
    """Build a simple baseline CNN."""
    # YOUR CODE HERE
    model = None  # Replace with your model
    return model

simple_cnn = build_simple_cnn()
simple_cnn.summary()

### Bonus (not marked): Task 6: Build Deep CNN with Regularization

Create a deeper CNN with:
- Data augmentation layer at the start
- 3 blocks, each with 2 Conv2D layers
- BatchNormalization after each Conv2D
- Dropout (0.25) after each block
- Dense classifier with 256 units and Dropout (0.5)

In [ ]:
def build_deep_cnn():
    """Build a deeper CNN with regularization."""
    # YOUR CODE HERE
    model = None  # Replace with your model
    return model

deep_cnn = build_deep_cnn()
print(f"Deep CNN Total Parameters: {deep_cnn.count_params():,}")

### Bonus (not marked): Task 7: Build Transfer Learning Model

Create a transfer learning model using MobileNetV2:
- Resize input from 32x32 to 96x96
- Use pre-trained MobileNetV2 (frozen)
- Add GlobalAveragePooling2D
- Custom classifier with Dropout

In [ ]:
def build_transfer_model():
    """Build a transfer learning model using MobileNetV2."""
    # YOUR CODE HERE
    # Hint: Use applications.MobileNetV2 with weights='imagenet', include_top=False
    model = None
    base_model = None
    return model, base_model

transfer_model, base_model = build_transfer_model()
print(f"Transfer Model Parameters: {transfer_model.count_params():,}")

## Part 5: Training the Models

### Task 8: Define Callbacks and Train Models

In [ ]:
# Define callbacks
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        )
    ]

In [ ]:
# Train Simple CNN
print("="*50)
print("Training Simple CNN (Baseline)")
print("="*50)

# YOUR CODE HERE
# Compile and train simple_cnn
# Use optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']
# Train for 30 epochs with batch_size=64

history_simple = None  # Replace with training history

In [ ]:
# Train Deep CNN
print("="*50)
print("Training Deep CNN with Regularization")
print("="*50)

# YOUR CODE HERE
# Compile and train deep_cnn
# Train for 50 epochs with batch_size=64

history_deep = None  # Replace with training history

In [ ]:
# Train Transfer Learning Model
print("="*50)
print("Training Transfer Learning Model")
print("="*50)

# YOUR CODE HERE
# Compile with learning_rate=1e-3
# Train for 20 epochs with batch_size=64

history_transfer = None  # Replace with training history

## Part 6: Model Evaluation and Comparison

### Task 9: Evaluate and Compare Models

In [ ]:
# YOUR CODE HERE
# Evaluate all three models on the test set
# Print loss and accuracy for each



In [ ]:
# YOUR CODE HERE
# Create plots comparing:
# 1. Validation accuracy over epochs for all three models
# 2. Validation loss over epochs for all three models



## Part 7: Detailed Analysis of Best Model

### Task 10: Confusion Matrix and Classification Report

In [ ]:
# YOUR CODE HERE
# 1. Select the best model based on test accuracy
# 2. Generate predictions on the test set
# 3. Create a confusion matrix visualization
# 4. Print the classification report



### Bonus (not marked): Task 11: Visualize Predictions

In [ ]:
# YOUR CODE HERE
# Display 15 random test images with their true labels, predicted labels, and confidence
# Color correct predictions in green, wrong predictions in red



## Part 8: Error Analysis

### Task 12: Analyze Misclassifications

In [ ]:
# YOUR CODE HERE
# 1. Find all misclassified samples
# 2. Count the most common confusion pairs (true class → predicted class)
# 3. Print the top 10 confused class pairs



In [ ]:
# YOUR CODE HERE
# Visualize 10 misclassified examples



## Part 9: Conclusions

### Task 13: Write Your Conclusions

Based on your experiments, answer the following questions:

1. Which model performed best and why?
2. What effect did data augmentation have on the deep CNN?
3. Which class pairs were most commonly confused? Why might this be?
4. What would you recommend for a production system?

*Write your answers below:*

**Your Conclusions:**

1. Best model: 

2. Data augmentation effect: 

3. Confused class pairs: 

4. Production recommendation: 


---

## Bonus Exercises

### Bonus (not marked): Exercise 1: Fine-tune the Transfer Learning Model
Unfreeze the last 20 layers of MobileNetV2 and fine-tune with a lower learning rate. Does this improve performance?

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 2: Class-Specific Analysis
Which class has the lowest accuracy? Investigate why and propose solutions.

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 3: Try Different Architectures
Replace MobileNetV2 with ResNet50 or EfficientNetB0. Compare the results.

In [ ]:
# YOUR CODE HERE



### Bonus (not marked): Exercise 4: Implement Test-Time Augmentation (TTA)
Apply augmentation at test time and average predictions. Does this improve accuracy?

In [ ]:
# YOUR CODE HERE

